# AutoEIT: Automated Scoring Engine for Spanish Elicited Imitation Tasks

**Applicant:** Anshika Choudhary  
**Target Project:** Automated scoring system for elicited imitation task responses

## 1. Introduction & Methodology
The Spanish Elicited Imitation Task (EIT) measures global oral proficiency, but human scoring is labor-intensive. Standard Large Language Models (LLMs) often suffer from hallucination and produce inconsistent scores across sessions. 

To solve this, this notebook implements a **Deterministic, Hybrid Scoring Engine** combining rule-based heuristics with machine learning embeddings. 

### Approach to the Task
1. **Text Preprocessing (The Cleaner):** The EIT rubric explicitly states that hesitations and false starts must not be penalized. Furthermore, unintelligible garble is marked as "XXX". I built a regex engine to strip these transcriber artifacts (e.g., `[pause]`, `xxx`, `ma-`) and target syllable counts (e.g., `(7)`) to prevent them from artificially lowering the semantic score.
2. **Semantic Similarity (ML Component):** Using `sentence-transformers` (`paraphrase-multilingual-MiniLM-L12-v2` via PyTorch), the engine converts both the target prompt and the cleaned transcription into high-dimensional vectors to calculate their Cosine Similarity.
3. **Rubric Application (Rule-Based):** The mathematical similarity is deterministically mapped to the 0-4 EIT scale:
    * **Score 4:** Awarded for exact string matching.
    * **Score 3:** Similarity $\ge$ 0.90 (meaning preserved with minor synonymous/grammatical changes).
    * **Score 2:** Similarity $\ge$ 0.70 (more than half of idea units preserved).
    * **Score 1:** Similarity $\ge$ 0.40 (half or fewer idea units represented).
    * **Score 0:** Awarded if the transcription is empty, garbled, or contains fewer than 2 words, aligning with the "minimal repetition/item abandoned" criteria.

In [1]:
# --- 1. Environment Setup & Dependencies ---
import pandas as pd
from pathlib import Path
import re
import torch
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import warnings
warnings.filterwarnings('ignore')

# Guarantee Deterministic Output for Reproducibility
RANDOM_SEED = 42
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

print("Loading the Multilingual NLP model...")
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
print("Model loaded successfully.")

Loading the Multilingual NLP model...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully.


In [2]:
# --- 2. Core Scoring Logic & Text Preprocessing ---

def clean_transcription(text):
    """
    Strips out transcriber artifacts to evaluate only the final produced speech.
    """
    if pd.isna(text): return ""
    text = str(text).lower()
    
    text = re.sub(r'\(\d+\)', '', text) # Remove target syllable counts like (7)
    text = re.sub(r'\[.*?\]', '', text) # Remove transcriber tags like [pause], [gibberish]
    text = re.sub(r'\bxxx\b', '', text) # Remove unintelligible markers
    text = re.sub(r'\b\w+-', '', text)  # Remove false starts (e.g., "ma-")
    text = re.sub(r'[^\w\s]', '', text) # Remove punctuation for clean matching
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

def calculate_semantic_score(target, transcription):
    """
    Applies the hybrid EIT rubric to return a discrete score from 0 to 4.
    """
    target_clean = clean_transcription(target)
    transcription_clean = clean_transcription(transcription)
    
    # Rule: Score 0 for silence or abandoned items
    if not transcription_clean or len(transcription_clean.split()) < 2:
        return 0
        
    # Rule: Score 4 for exact matches
    if target_clean == transcription_clean:
        return 4
        
    # ML Semantic Evaluation
    embeddings = model.encode([target_clean, transcription_clean])
    sim_score = cosine_similarity([embeddings[0]], [embeddings[1]])[0][0]
    
    # Rule: Map similarity thresholds to EIT Rubric
    if sim_score >= 0.90: return 3
    elif sim_score >= 0.70: return 2
    elif sim_score >= 0.40: return 1
    else: return 0

In [3]:
# --- 3. Data Pipeline Execution ---

DATA_DIR = Path("../data")
INPUT_FILE = DATA_DIR / "AutoEIT Sample Transcriptions for Scoring.xlsx"
OUTPUT_FILE = DATA_DIR / "AutoEIT_Scored_Transcriptions.xlsx"

def process_eit_dataset(input_path, output_path):
    """
    Ingests the Excel file, skips informational tabs, applies the scoring engine, and exports.
    """
    try:
        data_dict = pd.read_excel(input_path, sheet_name=None, engine='openpyxl')
        processed_dict = {}
        
        for sheet_name, df in data_dict.items():
            processed_df = df.copy()
            
            # Target only participant sheets containing the required columns
            if 'Transcription Rater 1' in processed_df.columns and 'Stimulus' in processed_df.columns:
                print(f"Applying NLP scoring to participant: {sheet_name}...")
                
                scores = []
                for index, row in processed_df.iterrows():
                    target = row['Stimulus']
                    transcription = row['Transcription Rater 1']
                    scores.append(calculate_semantic_score(target, transcription))
                    
                processed_df['Score'] = scores
                
            processed_dict[sheet_name] = processed_df
            
        # Export
        with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
            for sheet_name, df in processed_dict.items():
                df.to_excel(writer, sheet_name=sheet_name, index=False)
                
        print(f"\nPipeline Complete. Scored data saved to: {output_path}")
        return processed_dict
        
    except Exception as e:
        print(f"Pipeline failed: {e}")
        return None

# Execute the pipeline
final_scored_data = process_eit_dataset(INPUT_FILE, OUTPUT_FILE)

Applying NLP scoring to participant: 38001-1A...
Applying NLP scoring to participant: 38002-2A...
Applying NLP scoring to participant: 38004-2A...
Applying NLP scoring to participant: 38006-2A...

Pipeline Complete. Scored data saved to: ..\data\AutoEIT_Scored_Transcriptions.xlsx


## 2. Evaluating the Output
Because ground-truth human consensus scores are not provided in this specific test dataset, traditional accuracy metrics (like a confusion matrix) cannot be computed. 

Instead, to validate the engine, the script below extracts specific sentence pairs computed by the model for scores `0, 2, 3, and 4`. By displaying the raw transcription alongside the "Cleaned Transcription", we can manually verify that the regex engine correctly handles disfluencies without penalization, and that the ML model accurately captures semantic drift.

In [4]:
# --- 4. Output Evaluation & Edge Case Analysis ---
print("--- EIT Scoring Engine Qualitative Evaluation ---\n")

if final_scored_data:
    # Combine all scored participant data into a single DataFrame for analysis
    all_scored_df = pd.concat([df for name, df in final_scored_data.items() if 'Score' in df.columns])

    def display_score_example(score_val, description):
        example = all_scored_df[all_scored_df['Score'] == score_val]
        if not example.empty:
            row = example.iloc[0] # Grab the first instance
            print(f"Evaluating Score {score_val} ({description}):")
            print(f"  Raw Target:        {row['Stimulus']}")
            print(f"  Raw Transcription: {row['Transcription Rater 1']}")
            print(f"  Cleaned Target:    {clean_transcription(row['Stimulus'])}")
            print(f"  Cleaned Output:    {clean_transcription(row['Transcription Rater 1'])}\n")

    # Display examples spanning the rubric
    display_score_example(4, "Exact Match")
    display_score_example(3, "Meaning Preserved, Minor Changes")
    display_score_example(2, "More than Half Preserved, Inexact")
    display_score_example(0, "Minimal/Garbled Output")

--- EIT Scoring Engine Qualitative Evaluation ---

Evaluating Score 4 (Exact Match):
  Raw Target:        Quiero cortarme el pelo (7)
  Raw Transcription: Quiero cortarme el pelo
  Cleaned Target:    quiero cortarme el pelo
  Cleaned Output:    quiero cortarme el pelo

Evaluating Score 3 (Meaning Preserved, Minor Changes):
  Raw Target:        ¿Qué dice usted que va a hacer hoy? (9)
  Raw Transcription: Que dices ustedes se que van a hacer hoy?
  Cleaned Target:    qué dice usted que va a hacer hoy
  Cleaned Output:    que dices ustedes se que van a hacer hoy

Evaluating Score 2 (More than Half Preserved, Inexact):
  Raw Target:        Dudo que sepa manejar muy bien (10)
  Raw Transcription: Dudo que sepa manajar bien
  Cleaned Target:    dudo que sepa manejar muy bien
  Cleaned Output:    dudo que sepa manajar bien

Evaluating Score 0 (Minimal/Garbled Output):
  Raw Target:        El carro lo tiene Pedro (8)
  Raw Transcription: E-[gibberish] perro
  Cleaned Target:    el carro lo tie